# Notebook 20 — TrOCR + EXTERNAL Formulary Normalisation

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

DATA_ROOT = Path("../data/pharmacy_lk")
TRAIN_CSV = DATA_ROOT / "splits/train.csv"
TEST_CSV  = DATA_ROOT / "splits/test.csv"
LABEL_COL = "medicine_name"; IMG_COL = "image_filename"
TROCR_CKPT = Path("../checkpoints/benchmark_trocr/best")
PROC = "microsoft/trocr-small-handwritten"
FORMULARY_CSV = Path("../data/formulary/drug_names.csv")   # <-- you provide this

Using device: mps


## 1. Metrics + edit distance

In [2]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]

def metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs)
    em = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot/max(chars,1), "EM": em/len(refs), "n": len(refs)}

## 2. Build lexicons: training-only vs external formulary

**Required format for ../data/formulary/drug_names.csv** — one column named `drug_name`:
```
drug_name
augmentin
amoxicillin
panadol
paracetamol
...
```
Lowercase, one name per row, a few thousand rows ideally. Brand + generic names both help.

In [3]:
train_names = sorted(set(pd.read_csv(TRAIN_CSV)[LABEL_COL].astype(str).str.strip().str.lower()))
print(f"training-only lexicon: {len(train_names)} names")

if FORMULARY_CSV.exists():
    ext = pd.read_csv(FORMULARY_CSV)
    col = "drug_name" if "drug_name" in ext.columns else ext.columns[0]
    external = sorted(set(ext[col].astype(str).str.strip().str.lower()) - {""})
    # a real external formulary should also be UNIONED with training names
    external = sorted(set(external) | set(train_names))
    FORMULARY_IS_REAL = True
    print(f"EXTERNAL formulary loaded: {len(external)} names (real)")
else:
    # DEMO ONLY — ceiling illustration, peeks at test, NOT a valid result
    test_names = sorted(set(pd.read_csv(TEST_CSV)[LABEL_COL].astype(str).str.strip().str.lower()))
    external = sorted(set(train_names) | set(test_names))
    FORMULARY_IS_REAL = False
    print(f"!! No real formulary found. Using DEMO (train+test) = {len(external)} names.")
    print("!! DEMO peeks at test labels — it shows the UPPER BOUND only, not a valid result.")

def build_index(names):
    by_len = defaultdict(list)
    for w in names: by_len[len(w)].append(w)
    return set(names), by_len

train_set, train_idx = build_index(train_names)
ext_set, ext_idx = build_index(external)

def nearest(word, by_len, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd

def snap(word, by_len, tau=0.4):
    e, d = nearest(word, by_len)
    return e if (e is not None and d/max(len(word),1) <= tau) else word

training-only lexicon: 1210 names
EXTERNAL formulary loaded: 4779 names (real)


## 3. TrOCR raw predictions on test

In [4]:
processor = TrOCRProcessor.from_pretrained(PROC)
trocr = VisionEncoderDecoderModel.from_pretrained(TROCR_CKPT).to(DEVICE).eval()

def trocr_one(pil):
    pv = processor(pil.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        ids = trocr.generate(pv, max_new_tokens=24)
    return processor.decode(ids[0], skip_special_tokens=True).strip().lower()

df = pd.read_csv(TEST_CSV).dropna(subset=[LABEL_COL])
raw, refs, files = [], [], []
print("running TrOCR on test...")
for _, r in df.iterrows():
    fn = str(r[IMG_COL]); p = DATA_ROOT / "images" / fn
    if not p.exists(): continue
    raw.append(trocr_one(Image.open(p)))
    refs.append(str(r[LABEL_COL]).strip().lower()); files.append(fn)
print(f"{len(raw)} predictions")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

running TrOCR on test...
791 predictions


## 4. Compare: raw vs training-lexicon vs external-formulary

In [5]:
pred_train = [snap(p, train_idx) for p in raw]
pred_ext   = [snap(p, ext_idx)   for p in raw]

print("TEST comparison:")
for name, preds in [("TrOCR raw", raw),
                    ("TrOCR + training lexicon", pred_train),
                    ("TrOCR + external formulary", pred_ext)]:
    m = metrics(preds, refs)
    print(f"  {name:30s}: CER {m['CER']:.4f} | EM {m['EM']:.4f}")

# seen/unseen (seen = name in TRAINING data)
seen_mask = [r in train_set for r in refs]
def split_em(preds):
    s = [(p, r) for p, r, se in zip(preds, refs, seen_mask) if se]
    u = [(p, r) for p, r, se in zip(preds, refs, seen_mask) if not se]
    se = metrics([p for p,_ in s], [r for _,r in s])["EM"] if s else 0
    ue = metrics([p for p,_ in u], [r for _,r in u])["EM"] if u else 0
    return se, ue, len(s), len(u)

print("\nseen / unseen EM:")
for name, preds in [("TrOCR raw", raw), ("+ training lexicon", pred_train), ("+ external formulary", pred_ext)]:
    se, ue, ns, nu = split_em(preds)
    print(f"  {name:24s}: seen {se:.3f} (n={ns}) | unseen {ue:.3f} (n={nu})")

# coverage: how many UNSEEN test names are actually in the external formulary?
unseen_names = sorted({r for r, se in zip(refs, seen_mask) if not se})
covered = sum(1 for n in unseen_names if n in ext_set)
print(f"\nexternal formulary covers {covered}/{len(unseen_names)} unseen test names "
      f"({covered/max(len(unseen_names),1):.1%})")
if not FORMULARY_IS_REAL:
    print("!! Reminder: DEMO formulary — these numbers are an UPPER BOUND, not a valid result.")

TEST comparison:
  TrOCR raw                     : CER 0.2159 | EM 0.5689
  TrOCR + training lexicon      : CER 0.2199 | EM 0.6119
  TrOCR + external formulary    : CER 0.2044 | EM 0.6587

seen / unseen EM:
  TrOCR raw               : seen 0.701 (n=615) | unseen 0.108 (n=176)
  + training lexicon      : seen 0.779 (n=615) | unseen 0.028 (n=176)
  + external formulary    : seen 0.774 (n=615) | unseen 0.256 (n=176)

external formulary covers 156/172 unseen test names (90.7%)


## 5. Interpretation
- REAL formulary, external > training lexicon on UNSEEN: the external list recovers names the
  training lexicon could not -> genuine improvement, directly addresses the open-vocab gap.
- REAL formulary, no improvement: the external list does not cover the unseen names that
  appear in this test set (or snaps wrongly) -> honest negative; report coverage as the reason.
- The 'coverage' line is the key diagnostic: the approach can only help on unseen names that
  are actually present in the external formulary.